In [1]:
!pip install sequence_align

  Using cached sequence_align-0.3.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (8.0 kB)
Using cached sequence_align-0.3.0-cp37-abi3-macosx_11_0_arm64.whl (200 kB)


In [ ]:
from sequence_align.pairwise import needleman_wunsch

def align(seq_a, seq_b, match_score=1, mismatch_score=-1, indel_score=-1.0):

	aligned_seq_a, aligned_seq_b = needleman_wunsch(
		seq_a,
		seq_b,
		match_score=1.0,
		mismatch_score=-1.0,
		indel_score=-1.0,
		gap="_",
	)

	score_seq = []
	for x, y in zip(aligned_seq_a, aligned_seq_b):
		if x == y:
			score_seq.append(0)
		elif x == "_" or y == "_":
			score_seq.append(0.5)
		else:
			score_seq.append(1)

	tot_score = sum(score_seq)/len(score_seq)

	return aligned_seq_a, aligned_seq_b, score_seq, tot_score

In [26]:
files_transcriber = ["../data/new_output/01_ParlaBOA_E.conll.form", "../data/new_output/01_ParlaBOA_M.conll.form", "../data/new_output/01_ParlaBOB_N.conll.form", "../data/new_output/01_PastiA_O.conll.form", "../data/new_output/01_PastiA_R.conll.form", "../data/new_output/01_PastiB_V.conll.form", "../data/new_output/01_StraParlaA_I.conll.form", "../data/new_output/01_StraParlaA_L.conll.form", "../data/new_output/01_StraParlaB_A.conll.form", "../data/new_output/01_StraParlaC_S.conll.form", "../data/new_output/01_StraParlaC_U.conll.form", "../data/new_output/02_ParlaBOA_I.conll.form", "../data/new_output/02_ParlaBOA_L.conll.form", "../data/new_output/02_ParlaBOB_S.conll.form", "../data/new_output/02_PastiA_E.conll.form", "../data/new_output/02_PastiA_M.conll.form", "../data/new_output/02_PastiB_U.conll.form", "../data/new_output/02_StraParlaA_A.conll.form", "../data/new_output/02_StraParlaA_V.conll.form", "../data/new_output/02_StraParlaB_O.conll.form", "../data/new_output/02_StraParlaB_R.conll.form", "../data/new_output/02_StraParlaC_N.conll.form", "../data/new_output/03_ParlaBOA.conll.form", "../data/new_output/03_ParlaBOB.conll.form", "../data/new_output/03_PastiA.conll.form", "../data/new_output/03_PastiB.conll.form", "../data/new_output/03_StraParlaA.conll.form", "../data/new_output/03_StraParlaB.conll.form", "../data/new_output/03_StraParlaC.conll.form"]

files_whisper = []
files_output = []
for filename in files_transcriber:
    filename_split = filename.split("/")
    prefix, data, folder, fname = filename_split
    corresponding_fname = fname.split("_")[1].split(".")[0]
    output_fname = fname.split(".")[0]

    files_whisper.append('/'.join([prefix, data, 'whisper_output', f"{corresponding_fname}.vert.csv"]))
    files_output.append('/'.join([prefix, data, 'whisper_output_aligned', f"{output_fname}.csv"]))


for file_whisper, file_transcriber, file_output in zip(files_whisper, files_transcriber, files_output):
    fout = open(file_output, "w")
    tokens_whisper = []
    with open(file_whisper) as fin:
        for line in fin:
            line = line.strip().split()
            tokens_whisper.append(line[-1])

    tokens_transcriber = []
    with open(file_transcriber) as fin:
        fin.readline()
        for line in fin:
            line = line.strip().split()
            tokens_transcriber.append(line[-1])

    tokens_whisper = [''.join(c for c in x if not c in ["?", "!", ".", ","]) for x in tokens_whisper]

    aligned_whisper, aligned_transcriber, _, _ = align(tokens_whisper, tokens_transcriber)
    new_aligned_whisper = []
    new_aligned_transcriber = []
    for x, y in zip(aligned_transcriber, aligned_whisper):
        if x == "_" or y == "_" or x == y:
            new_aligned_whisper.append(y)
            new_aligned_transcriber.append(x)
        else:
            new_aligned_whisper.append(y)
            new_aligned_whisper.append("_")

            new_aligned_transcriber.append("_")
            new_aligned_transcriber.append(x)


    matches = []
    for x, y in zip(new_aligned_transcriber, new_aligned_whisper):
        if x == "_":
            matches.append(0)
        elif y == "_":
            matches.append(0)
        elif x == y:
            matches.append(1)


    for i, (x, y, z) in enumerate(zip(new_aligned_whisper, new_aligned_transcriber, matches)):
        print(f"{i}\t{x}\t{y}\t{z}", file=fout)

    fout.close()